# Window Functions for Analytics

In previous lessons, we used `GROUP BY` to aggregate data (like finding the total sales per department). However, `GROUP BY` has a major limitation: **it squashes your rows together**. If you group by department, you lose the individual employee details. 

What if you want to see an individual employee's name, their specific salary, AND the department average all on the same row? 

Enter **Window Functions**. They allow you to perform calculations across a specific set of rows (a "window") *without* collapsing the original rows. 

Let's set up our Python sandbox with some sales data over time!

In [1]:
import sqlite3
import pandas as pd

# 1. Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create a Daily Sales table
cursor.executescript("""
CREATE TABLE SalesData (
    sale_id INTEGER PRIMARY KEY,
    rep_name TEXT,
    region TEXT,
    sale_date DATE,
    amount DECIMAL(10, 2)
);

INSERT INTO SalesData (rep_name, region, sale_date, amount) VALUES 
('Alice', 'North', '2023-11-01', 500),
('Alice', 'North', '2023-11-02', 800),
('Bob', 'South', '2023-11-01', 300),
('Bob', 'South', '2023-11-02', 450),
('Bob', 'South', '2023-11-03', 200),
('Charlie', 'North', '2023-11-01', 1200),
('Charlie', 'North', '2023-11-03', 600);
""")

print("✅ Database ready! Daily Sales data loaded.")

✅ Database ready! Daily Sales data loaded.


# 1. The `OVER()` Clause (The Basics)
The magic word for Window Functions is `OVER()`. If you use an aggregate function like `SUM()` but follow it with `OVER()`, it tells the database: *"Calculate the sum, but attach the grand total to every single row without squashing them."*

In [2]:
# Show me every individual sale, but add a column showing the company's Grand Total.
query_over = """
SELECT 
    rep_name, 
    sale_date, 
    amount, 
    SUM(amount) OVER() as company_grand_total
FROM SalesData;
"""
print("--- Standard OVER() (No squashing!) ---")
display(pd.read_sql_query(query_over, conn))

--- Standard OVER() (No squashing!) ---


,rep_name,sale_date,amount,company_grand_total
0,Alice,2023-11-01,500,4050
1,Alice,2023-11-02,800,4050
2,Bob,2023-11-01,300,4050
3,Bob,2023-11-02,450,4050
4,Bob,2023-11-03,200,4050
5,Charlie,2023-11-01,1200,4050
6,Charlie,2023-11-03,600,4050


# 2. `PARTITION BY` (Grouping within the Window)
Usually, we don't just want the grand total. We want totals grouped by categories. `PARTITION BY` works exactly like `GROUP BY`, but again, it keeps the rows intact. It creates "mini-windows" based on a category.

In [3]:
# Show me every individual sale, but add a column showing the TOTAL for that specific REGION.
query_partition = """
SELECT 
    rep_name, 
    region, 
    amount, 
    SUM(amount) OVER(PARTITION BY region) as region_total
FROM SalesData;
"""
print("\n--- PARTITION BY (Grouped totals kept on individual rows) ---")
display(pd.read_sql_query(query_partition, conn))


--- PARTITION BY (Grouped totals kept on individual rows) ---


,rep_name,region,amount,region_total
0,Alice,North,500,3100
1,Alice,North,800,3100
2,Charlie,North,1200,3100
3,Charlie,North,600,3100
4,Bob,South,300,950
5,Bob,South,450,950
6,Bob,South,200,950


# 3. `ORDER BY` (Creating Running Totals)
If you add `ORDER BY` inside the `OVER()` clause, the window changes from a static group into a **moving window**. It calculates the sum *up to the current row*, creating a cumulative running total!

In [4]:
# Create a running total of sales for each sales rep over time.
query_running_total = """
SELECT 
    rep_name, 
    sale_date, 
    amount, 
    SUM(amount) OVER(PARTITION BY rep_name ORDER BY sale_date) as rep_running_total
FROM SalesData;
"""
print("\n--- ORDER BY (Cumulative Running Totals) ---")
display(pd.read_sql_query(query_running_total, conn))


--- ORDER BY (Cumulative Running Totals) ---


,rep_name,sale_date,amount,rep_running_total
0,Alice,2023-11-01,500,500
1,Alice,2023-11-02,800,1300
2,Bob,2023-11-01,300,300
3,Bob,2023-11-02,450,750
4,Bob,2023-11-03,200,950
5,Charlie,2023-11-01,1200,1200
6,Charlie,2023-11-03,600,1800


*(Notice how Alice's running total goes from 500 on Nov 1st, to 1300 on Nov 2nd!)*

# 4. Ranking Functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`)
Window functions aren't just for math; they are amazing for ranking data.

* **`ROW_NUMBER()`**: Gives a unique 1, 2, 3, 4 to every row in the partition.
* **`RANK()`**: Gives rankings, but if there's a tie for 1st place, the next person gets 3rd place (1, 1, 3).
* **`DENSE_RANK()`**: Gives rankings, but doesn't skip numbers after ties (1, 1, 2).

In [5]:
# Rank the sales reps in each region based on their single largest sale.
query_rank = """
SELECT 
    rep_name, 
    region, 
    amount, 
    RANK() OVER(PARTITION BY region ORDER BY amount DESC) as sales_rank
FROM SalesData;
"""
print("\n--- Ranking within Regions ---")
display(pd.read_sql_query(query_rank, conn))


--- Ranking within Regions ---


,rep_name,region,amount,sales_rank
0,Charlie,North,1200,1
1,Alice,North,800,2
2,Charlie,North,600,3
3,Alice,North,500,4
4,Bob,South,450,1
5,Bob,South,300,2
6,Bob,South,200,3


# 5. `LAG` and `LEAD` (Time Travel)
As a Data Scientist, you will constantly need to calculate "Month-over-Month" or "Day-over-Day" changes. 

* **`LAG()`**: Looks backward and grabs data from the *previous* row.
* **`LEAD()`**: Looks forward and grabs data from the *next* row.

In [6]:
# For each rep, show their sale today, and their sale from the previous day so we can compare.
query_lag = """
SELECT 
    rep_name, 
    sale_date, 
    amount as sales_today,
    LAG(amount) OVER(PARTITION BY rep_name ORDER BY sale_date) as sales_yesterday
FROM SalesData;
"""
print("\n--- LAG() (Looking at the previous row) ---")
display(pd.read_sql_query(query_lag, conn))

# Clean up
conn.close()


--- LAG() (Looking at the previous row) ---


,rep_name,sale_date,sales_today,sales_yesterday
0,Alice,2023-11-01,500,NaN
1,Alice,2023-11-02,800,500.0
2,Bob,2023-11-01,300,NaN
3,Bob,2023-11-02,450,300.0
4,Bob,2023-11-03,200,450.0
5,Charlie,2023-11-01,1200,NaN
6,Charlie,2023-11-03,600,1200.0


*(Notice how the first day for every rep shows `None` / `NaN` for `sales_yesterday` because there is no previous day to look at!)*

## Real-World Use Case or Analogy:
Think of Window Functions vs. `GROUP BY` like comparing a **Bank Statement** to a **Tax Return**:

* **`GROUP BY` (The Tax Return)**: At the end of the year, the government doesn't care about every single time you bought coffee. They just want you to squash everything together and give them one final number: `SUM(Income)` and `SUM(Expenses)`. You lose all the daily details.
* **Window Functions (The Bank Statement)**: When you log into your banking app, you see a list of every single transaction you made (the individual rows). Next to the $5.00 coffee purchase, there is a column showing your `Current Account Balance` (The Running Total). The app didn't squash your transactions together; it calculated a moving window over your history so you could see the math happening line-by-line.

---